Imports

In [1]:
from pathlib import Path

import pandas as pd

Define path to a version of caf.ml model outputs

In [18]:
model_version = '1v2+'
results_path = Path(r'I:\NorMITs NorCOM\AECOM working\v6') / model_version

Look at validation of model by area type to make sure the accuracy result is not spatially biased

In [19]:
# read in model predictions and actual values
prediction = pd.read_csv(results_path / 'output' / 'final_predictions.csv')
actual = pd.read_csv(results_path / 'output' / 'validate.csv')

In [20]:
# combine the results 
overlap_columns = [col for col in prediction.columns if col in actual.columns]
combined = pd.merge(prediction, actual, on=overlap_columns, how='left')

In [21]:
combined

,psuid,householdid,surveyyear,hholdua_b01id,predicted_target_column,1v2+_car
0,2011000029,2011000348,2011,340,2,2
1,2006000153,2006002087,2006,400,1,1
2,2006000609,2006008257,2006,400,1,1
3,2002000489,2002006257,2002,481,2,2
4,2009000444,2009005982,2009,430,2,2
...,...,...,...,...,...,...
13272,2007000440,2007005981,2007,250,1,2
13273,2003000099,2003001376,2003,500,2,1
13274,2008000595,2008007823,2008,281,2,2
13275,2012000309,2012003963,2012,410,2,2


In [22]:
# read in validation dataset to get area type of household
test = pd.read_csv(results_path / 'output' / 'test.csv')

In [23]:
data = pd.merge(combined, test[['householdid', 'tfn_at_v1']], on='householdid', how='left')

In [24]:
data

,psuid,householdid,surveyyear,hholdua_b01id,predicted_target_column,1v2+_car,tfn_at_v1
0,2011000029,2011000348,2011,340,2,2,5
1,2006000153,2006002087,2006,400,1,1,5
2,2006000609,2006008257,2006,400,1,1,5
3,2002000489,2002006257,2002,481,2,2,6
4,2009000444,2009005982,2009,430,2,2,8
...,...,...,...,...,...,...,...
13272,2007000440,2007005981,2007,250,1,2,7
13273,2003000099,2003001376,2003,500,2,1,6
13274,2008000595,2008007823,2008,281,2,2,8
13275,2012000309,2012003963,2012,410,2,2,4


In [25]:
data['success'] = (data['predicted_target_column'] == data[f'{model_version}_car'])
data['total'] = True
data

,psuid,householdid,surveyyear,hholdua_b01id,predicted_target_column,1v2+_car,tfn_at_v1,success,total
0,2011000029,2011000348,2011,340,2,2,5,True,True
1,2006000153,2006002087,2006,400,1,1,5,True,True
2,2006000609,2006008257,2006,400,1,1,5,True,True
3,2002000489,2002006257,2002,481,2,2,6,True,True
4,2009000444,2009005982,2009,430,2,2,8,True,True
...,...,...,...,...,...,...,...,...,...
13272,2007000440,2007005981,2007,250,1,2,7,False,True
13273,2003000099,2003001376,2003,500,2,1,6,False,True
13274,2008000595,2008007823,2008,281,2,2,8,True,True
13275,2012000309,2012003963,2012,410,2,2,4,True,True


In [26]:
summary = data.groupby('tfn_at_v1').agg({'success': 'sum', 'total': 'sum'})
summary['area_type_accuracy'] = summary['success'] / summary['total']

In [27]:
summary

,success,total,area_type_accuracy
tfn_at_v1,,,
1,302,365,0.827397
2,607,867,0.700115
3,568,806,0.704715
4,971,1288,0.753882
5,2229,2967,0.751264
6,2890,3819,0.756743
7,1142,1527,0.747872
8,1270,1638,0.775336
